# RAG Application using Type Sense
- Notebook by Adam Lang
- Date: 8-18-2026

# Overview
- In this notebook we will create a RAG application using Type Sense. See docs: https://typesense.org/about

In [1]:
import typesense

## Create Type Sense Client and Data Schema for Vector DB

In [ ]:
## create type sense client
## the 'host' came from the Node endpoint config
client = typesense.Client({
    'nodes': [{
        'host': '<your host goes here>.a1.typesense.net', # for Type Sense cloud use xxx.a1.typesense.net -- can use 'local'
        'port': '443', # for Typesense cloud use 443 -- can use local 8080
        'protocol': 'https' # for Typesense cloud use https
    }],
    'api_key': '<your api key goes here>', ## if using local host don't need API key
    'connection_timeout_seconds': 2
})


## Create data schema -- we will use a jsonl file of books
books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet': True},
        {'name': 'publication_year', 'type': 'int32', 'facet': True},
        {'name': 'ratings_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'}
    ],
    'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

Note: "Books" collection was already created. 

In [3]:
## client
client

## Open jsonl books file in Read mode

In [4]:
## open jsonl file
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

## Create Search Parameters

In [5]:
## search parameters
search_parameters = {
    'q': "harry potter",
    'query_by': "title, authors",
    'sort_by': "ratings_count:desc",
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

### Search using Filter Criteria

In [6]:
## search parameters
search_parameters = {
    'q': "harry potter",
    'query_by': "title",
    'filter_by': 'publication_year:<1998',
    'sort_by': "publication_year:desc",
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

### Another search-filter criteria

In [7]:
## search params
search_parameters = {
    'q': 'experiment',
    'query_by': 'title',
    'sort_by': 'average_rating:desc',
}
## search collection
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 578730123365189753,
   'text_match_info': {'best_field_score': '1108091338753',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '578730123365189753',
    'tokens_matched': 1,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['Mahatma Gandhi'],
    'average_rating': 4.07,
    'id': '2649',
    'image_url': 'https://images.gr-assets.com/books/1320560971m/112803.jpg',
   

## LangChain + TypeSense + Groq LLM + RAG App
- you can also do this with LangChain

In [3]:
### LangChain + TypeSense + Groq LLM + RAG application
# from langchain_groq import GroqLLM
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense ## vector DB
from langchain_text_splitters import CharacterTextSplitter ## chunking/text splitting
from langchain_huggingface import HuggingFaceEmbeddings ## embedding models
from langchain_groq import ChatGroq





C:\Users\pytho\AppData\Local\Temp\ipykernel_26928\1052535793.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\pytho\OneDrive\Documents\Software_Engineering_Stuff\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setup GROQ LLM API key

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

## init groq LLM (set API key in environment)
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY")

In [9]:
## init groq llm API
from langchain.chat_models import init_chat_model

# llm=init_chat_model("groq:llama-3.3-70b-versatile") ## model being phased out on Aug 16, 2026
llm = init_chat_model("groq:openai/gpt-oss-120b",
                      temperature=0.1, max_tokens=1024) ## new model
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013181A34B90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001318006DA10>, model_name='openai/gpt-oss-120b', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=1024)

## Document Loader + Splitter
- see HuggingFaceEmbeddings docs: https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings

In [6]:
## load docs and split/chunk
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("notebooks/test_essay.txt", encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100) ## adjust this
docs = text_splitter.split_documents(documents)

Created a chunk of size 1203, which is longer than the specified 1000
Created a chunk of size 3736, which is longer than the specified 1000
Created a chunk of size 1989, which is longer than the specified 1000
Created a chunk of size 1760, which is longer than the specified 1000
Created a chunk of size 1208, which is longer than the specified 1000
Created a chunk of size 1132, which is longer than the specified 1000
Created a chunk of size 1176, which is longer than the specified 1000
Created a chunk of size 1449, which is longer than the specified 1000
Created a chunk of size 1298, which is longer than the specified 1000
Created a chunk of size 1262, which is longer than the specified 1000
Created a chunk of size 1463, which is longer than the specified 1000
Created a chunk of size 1091, which is longer than the specified 1000
Created a chunk of size 1058, which is longer than the specified 1000
Created a chunk of size 1125, which is longer than the specified 1000
Created a chunk of s

In [10]:
## create embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

c:\Users\pytho\OneDrive\Documents\Software_Engineering_Stuff\YTRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pytho\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 

## Document Search: LangChain + Typesense

In [ ]:
## create docsearch
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        'host': '<your host goes here>.a1.typesense.net', # for Type Sense cloud use xxx.a1.typesense.net -- can use 'local'
        'port': "443", # use 443 for Typesense Cloud
        'protocol': "https", # use https for Typesense cloud
        "typesense_api_key": '<your key goes here>',
        "typesense_collection_name": "lang-chain",

    }
)

## Query documents
- This is the source document: https://darioamodei.com/essay/the-adolescence-of-technology#3-the-odious-apparatus

In [14]:
query = "What is the odious apparatus?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Although biology is currently the most serious vector of attack, there are many other vectors and it is possible that a more dangerous one may emerge. The general principle is that without countermeasures, AI is likely to continuously lower the barrier to destructive activity on a larger and larger scale, and humanity needs a serious response to this threat.

3. The odious apparatus
Misuse for seizing power
The previous section discussed the risk of individuals and small organizations co-opting a small subset of the “country of geniuses in a datacenter” to cause large-scale destruction. But we should also worry—likely substantially more so—about misuse of AI for the purpose of wielding or seizing power, likely by larger and more established actors.29


## Concert to Retriever

In [15]:
## retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x0000013183262250>, search_kwargs={})

In [16]:
## now run query with retriever
query = "What is the odious apparatus?"
retriever.invoke(query)[0]

Document(metadata={'source': 'notebooks/test_essay.txt'}, page_content='Although biology is currently the most serious vector of attack, there are many other vectors and it is possible that a more dangerous one may emerge. The general principle is that without countermeasures, AI is likely to continuously lower the barrier to destructive activity on a larger and larger scale, and humanity needs a serious response to this threat.\n\n3. The odious apparatus\nMisuse for seizing power\nThe previous section discussed the risk of individuals and small organizations co-opting a small subset of the “country of geniuses in a datacenter” to cause large-scale destruction. But we should also worry—likely substantially more so—about misuse of AI for the purpose of wielding or seizing power, likely by larger and more established actors.29')